# ME 203 — Test 3: Classes and simulation

**35 minutes total for the Canvas quiz and this notebook · 22 points total**

Canvas questions: **10 points**. This notebook: **4 tasks × 3 points = 12 points**.

Name: **[Tega Ikoba]**

Suggested pacing: about 10 minutes for the quiz, 22 minutes for programming, and 3 minutes to check, save, and upload. You may divide your time differently.

- Follow the permitted-resource rules posted with the test in Canvas.
- Keep given function, class, and variable names. Calculate results; do not hard-code answers.
- Functions must work with other inputs satisfying the documented assumptions.
- Only NumPy is required. No `car.py`, external data files, animation, or Otter installation is needed.
- Inspect cells show outputs but are not autograder checks.
- Tasks have separate inputs. Task 3 provides a fallback if Task 2 is unfinished; using it does not earn Task 2 credit.
- Before the timer ends, restart the kernel, run all cells, save with outputs, and upload this `.ipynb` file to the programming upload question. Leave attempted code for partial credit.


In [1]:
import numpy as np

## Task 1 — Repair a class (3 points)

`MotionState` stores one object's vertical position `y` (m) and velocity `vy` (m/s).

Repair the constructor so that each instance stores the **supplied arguments** on `self`. Complete `values` so that it returns a tuple `(position, velocity)` using that instance's attributes. Do not return global variables.

The Inspect cell makes two instances, edits one, and prints both. They should retain independent state.

In [2]:
class MotionState:
    def __init__(self, y, vy):
        # TODO: store the supplied values on this instance.
        self.y = y
        self.vy = vy

    def values(self):
        # TODO: return this instance's (y, vy).
        return (self.y, self.vy)


In [3]:
# Inspect Task 1.
state_a = MotionState(y=2.0, vy=-1.0)
state_b = MotionState(y=5.0, vy=0.5)
state_a.y = 3.0
print("State A:", state_a.values())
print("State B:", state_b.values())


State A: (3.0, -1.0)
State B: (5.0, 0.5)


## Task 2 — Repair one Euler step (3 points)

Use the same **forward Euler** rule as Lecture Activity 6, simplified to vertical motion:

`y_next = y + vy * delta_t`

`vy_next = vy + ay * delta_t`

Position uses the velocity **at the beginning of the step**, not the updated velocity. `delta_t` is in **seconds**; `ay` is acceleration in m/s².

Repair the two update expressions below. Return `(y_next, vy_next)`. Use the arguments, with no hard-coded acceleration or time step. This task does not handle collisions or bouncing.

In [4]:
def simulate_one_step(y, vy, ay, delta_t):
    # TODO: repair both update expressions.
    vy_next = vy + ay * delta_t
    y_next = y + vy * delta_t
    return y_next, vy_next


In [5]:
# Inspect Task 2: calls use different inputs.
step_a = simulate_one_step(y=2.0, vy=-1.0, ay=-2.0, delta_t=0.5)
step_b = simulate_one_step(y=1.0, vy=2.0, ay=0.0, delta_t=0.25)
print("Step A (y, vy):", step_a)
print("Step B (y, vy):", step_b)


Step A (y, vy): (1.5, -2.0)
Step B (y, vy): (1.5, 2.0)


## Task 3 — Finish the simulation loop (3 points)

Complete the two marked lines in `simulate_n_steps`:

1. Call `step_function` using the **current** `y` and `vy` and the supplied `ay` and `delta_t`; assign its returned values back to `y` and `vy`.
2. Store the updated `(y, vy)` in the next output row.

The result has **n_steps + 1 rows**: initial state in row 0, followed by one row after each update. Column 0 is position, column 1 is velocity. Assume `n_steps` is a nonnegative integer and `delta_t > 0`.

The runner uses your Task 2 function by default. If Task 2 is unfinished, set `use_step_fallback = True` to use the supplied correct step function for Task 3 only. This does not change what is required in your loop.

In [6]:
def simulate_n_steps(y0, vy0, ay, delta_t, n_steps, step_function):
    trajectory = np.zeros((n_steps + 1, 2))
    y, vy = y0, vy0
    trajectory[0, :] = [y, vy]
    for i in range(n_steps):
        # TODO: replace these two lines.
        y, vy = step_function(y, vy, ay, delta_t)
        trajectory[i + 1, :] = [y, vy]
    return trajectory


In [7]:
# Inspect Task 3.
use_step_fallback = False

def provided_step(y, vy, ay, delta_t):
    return y + vy * delta_t, vy + ay * delta_t

chosen_step = provided_step if use_step_fallback else simulate_one_step
trajectory = simulate_n_steps(
    y0=2.0, vy0=0.0, ay=-2.0, delta_t=0.5,
    n_steps=2, step_function=chosen_step,
)
print("Rows: initial state, after step 1, after step 2")
print("Columns: y (m), vy (m/s)")
print(trajectory)


Rows: initial state, after step 1, after step 2
Columns: y (m), vy (m/s)
[[ 2.   0. ]
 [ 2.  -1. ]
 [ 1.5 -2. ]]


## Task 4 — Repair road sampling and explain what it misses (3 points)

As in Lab 6, road coordinates are in meters, vehicle speed is in m/s, and `dt_ms` is in **milliseconds**. The distance between samples must be:

`distance_step = velocity * (dt_ms / 1000)`

Repair the distance-step expression and the arguments to `np.interp`. The correct argument order is `np.interp(sample_positions, road_positions, road_heights)`. Use `np.arange` as supplied, excluding the road's final x coordinate. Assume increasing road x coordinates beginning at 0, a positive velocity, and `dt_ms > 0`.

Run the two supplied sampling intervals. Then explain the difference in one or two sentences. No plot is required.

In [8]:
road_x = np.array([0.0, 1.0, 1.25, 1.5, 3.0])
road_y = np.array([0.0, 0.0, 0.30, 0.0, 0.0])

def interpolate_road(X_r, Y_r, velocity, dt_ms):
    # TODO: repair the distance step and interpolation call.
    distance_step = velocity * (dt_ms / 1000)
    sample_x = np.arange(0.0, X_r[-1], distance_step)
    sample_y = np.interp(sample_x, X_r, Y_r)
    return sample_x, sample_y

fine_x, fine_y = interpolate_road(road_x, road_y, velocity=5.0, dt_ms=50.0)
coarse_x, coarse_y = interpolate_road(road_x, road_y, velocity=5.0, dt_ms=400.0)
print("50 ms positions:", fine_x)
print("50 ms heights:", fine_y)
print("400 ms positions:", coarse_x)
print("400 ms heights:", coarse_y)
print("Largest sampled heights:", np.max(fine_y), np.max(coarse_y))


50 ms positions: [0.   0.25 0.5  0.75 1.   1.25 1.5  1.75 2.   2.25 2.5  2.75]
50 ms heights: [0.  0.  0.  0.  0.  0.3 0.  0.  0.  0.  0.  0. ]
400 ms positions: [0. 2.]
400 ms heights: [0. 0.]
Largest sampled heights: 0.3 0.0


**Task 4 response:** State the largest sampled height for each interval. Explain why the 400 ms samples miss the bump, referring to their sampled x positions. Does increasing `dt_ms` remove the bump from the original road?

[in the 50 ms time step sample we see a max height of o.3 while the max height remains at 0 for the 400 ms sample. By increasing dt_ms, we can end up acidentally skipping over the bump which is what happened in this example above. the smaller the time step, the more accurate our data will be. However, computational speed will decrease when we have a smaller dt_ms becasue more data needs to be computed.]

## Submission

Restart, run all, and save. Upload this notebook with its outputs to the programming upload question before the timer ends. Answer the multiple-choice questions in Canvas. No other programming files are required.